In [1]:
#问题4：编写代码，测试特征归一化对各个分类器训练效率、预测精度和预测效率的影响。

In [2]:
# 公共初始化代码｜输出：数据来源、版本和独立输出目录
from pathlib import Path
from datetime import datetime
from time import perf_counter
import hashlib
import json
import platform
import sys
import uuid
import warnings
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from threadpoolctl import threadpool_limits, threadpool_info
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from sklearn.exceptions import ConvergenceWarning

PROJECT_DIR = Path(r"D:\Python for Data Analysis\notebooks\imbalanced_classification")
SOURCE_DIR = PROJECT_DIR / "runs/20261020_221259_7f297c"
Q3_SOURCE = PROJECT_DIR / "q3_runs/20261021_111602_87b2fd"
assert Path(sys.executable).resolve() == Path(r"D:\anaconda\envs\pda312\python.exe").resolve(), "请选择pda312内核"
source = json.loads((SOURCE_DIR / "results/manifest.json").read_text(encoding="utf-8"))
assert sklearn.__version__ == source["versions"]["scikit-learn"], "环境版本已变化，请先核对"
with np.load(SOURCE_DIR / "data/dataset_and_split.npz") as saved:
    X, y = saved["X"], saved["y"]
    train_idx, test_idx = saved["train_idx"], saved["test_idx"]
data_hash = hashlib.sha256(X.tobytes() + y.tobytes() + train_idx.tobytes() + test_idx.tobytes()).hexdigest()
assert data_hash == source["data_split_sha256"]
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
LABELS, MODEL_SEED = np.arange(10), source["model_seed"]
REPEATS, ORDER_SEED, THREAD_LIMIT = 5, 20261023, 1
Q4_DIR = PROJECT_DIR / "q4_runs" / (datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6])
Q4_DIR.mkdir(parents=True, exist_ok=False)
print("问题4 v1.0；Python", sys.version.split()[0], "；sklearn", sklearn.__version__)
print("训练/测试：", X_train.shape, X_test.shape, "；每设置计时重复：", REPEATS)
print("本题目录：", Q4_DIR)

问题4 v1.0；Python 3.12.13 ；sklearn 1.9.0
训练/测试： (3600, 24) (1200, 24) ；每设置计时重复： 5
本题目录： D:\Python for Data Analysis\notebooks\imbalanced_classification\q4_runs\20261021_122610_cb3b24


In [3]:
# 依赖：初始化；输出：Pipeline和固定计时协议
models = {
    "LogisticRegression": LogisticRegression(random_state=MODEL_SEED),
    "DecisionTree": DecisionTreeClassifier(random_state=MODEL_SEED),
    "KNN": KNeighborsClassifier(),
    "SVC": SVC(random_state=MODEL_SEED),
}
model_parameters = {name: model.get_params(deep=False) for name, model in models.items()}
assert model_parameters == json.loads((Q3_SOURCE / "q3_model_parameters.json").read_text(encoding="utf-8"))
pipelines = {}
for name, model in models.items():
    for setting in ("raw", "standardized"):
        scaler = "passthrough" if setting == "raw" else StandardScaler()
        pipelines[(name, setting)] = Pipeline([("scale", scaler), ("model", clone(model))])
print("模型超参数与第三题一致。raw=原始特征；standardized=StandardScaler。")
print("训练计时包括标准化fit_transform和模型fit；预测计时包括测试集transform和predict。")
print("计时不包括克隆模型、指标计算、打印和文件保存；预测耗时对应整个1200例测试集。")
print("每设置预热1次不计入统计；重复5次均重新拟合。每轮随机顺序，固定顺序种子。")
print("计时期间BLAS/OpenMP线程上限为1，退出代码块后恢复；不修改模型超参数。")

模型超参数与第三题一致。raw=原始特征；standardized=StandardScaler。
训练计时包括标准化fit_transform和模型fit；预测计时包括测试集transform和predict。
计时不包括克隆模型、指标计算、打印和文件保存；预测耗时对应整个1200例测试集。
每设置预热1次不计入统计；重复5次均重新拟合。每轮随机顺序，固定顺序种子。
计时期间BLAS/OpenMP线程上限为1，退出代码块后恢复；不修改模型超参数。


In [4]:
# 辅助代码：一次测量封装｜供下一核心单元调用；定义本身不运行实验
measurements, warning_rows, prediction_parts, class_parts, diagnostic_rows = [], [], [], [], []
def measure_once(name, setting, repeat, phase, order):
    pipe = clone(pipelines[(name, setting)])  # 计时外新建模型，不复用上次已拟合对象。
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        start = perf_counter()
        pipe.fit(X_train, y_train)
        fit_seconds = perf_counter() - start
        start = perf_counter()
        pred = pipe.predict(X_test)
        predict_seconds = perf_counter() - start
    for item in caught:
        warning_rows.append(dict(model=name, setting=setting, repeat=repeat, phase=phase,
                                 category=item.category.__name__, message=str(item.message),
                                 is_convergence_warning=issubclass(item.category, ConvergenceWarning)))
    if phase == "warmup":
        return
    estimator = pipe.named_steps["model"]
    diagnostic_rows.append(dict(model=name, setting=setting, repeat=repeat,
        n_iter=np.asarray(estimator.n_iter_).tolist() if hasattr(estimator, "n_iter_") else None,
        fit_status=int(estimator.fit_status_) if hasattr(estimator, "fit_status_") else None))
    measurements.append(dict(model=name, setting=setting, repeat=repeat, order=order,
        fit_seconds=fit_seconds, predict_seconds=predict_seconds,
        accuracy=accuracy_score(y_test, pred),
        macro_f1=f1_score(y_test, pred, labels=LABELS, average="macro", zero_division=0)))
    prediction_parts.append(pd.DataFrame(dict(model=name, setting=setting, repeat=repeat,
        sample_id=test_idx, y_true=y_test, y_pred=pred)))
    precision, recall, f1, support = precision_recall_fscore_support(
        y_test, pred, labels=LABELS, average=None, zero_division=0)
    class_parts.append(pd.DataFrame(dict(model=name, setting=setting, repeat=repeat,
        class_label=LABELS, support=support, precision=precision, recall=recall, f1=f1)))

In [5]:
# 依赖：测量函数；输出：进度和逐次真实计时记录
# 重新运行本单元会重做计时实验；先清空本单元产生的内存列表，避免重复累加。
measurements.clear(); warning_rows.clear(); prediction_parts.clear(); class_parts.clear(); diagnostic_rows.clear()
keys, rng = list(pipelines), np.random.default_rng(ORDER_SEED)
with threadpool_limits(limits=THREAD_LIMIT):
    controlled_threadpools = threadpool_info()
    for order, index in enumerate(rng.permutation(len(keys)), start=1):
        measure_once(*keys[index], repeat=0, phase="warmup", order=order)
    print("8种设置预热完成，下面开始正式计时。", flush=True)
    for repeat in range(1, REPEATS + 1):
        for order, index in enumerate(rng.permutation(len(keys)), start=1):
            measure_once(*keys[index], repeat=repeat, phase="measured", order=order)
        print(f"第{repeat}/{REPEATS}轮完成。", flush=True)
records = pd.DataFrame(measurements)
assert len(records) == len(keys) * REPEATS
display(records.round(6))
print("警告总数（含预热）：", len(warning_rows))
for name, setting, category, message in sorted({(w["model"], w["setting"], w["category"], w["message"]) for w in warning_rows}):
    print(f"{name}/{setting} [{category}]：{message}")

8种设置预热完成，下面开始正式计时。
第1/5轮完成。
第2/5轮完成。
第3/5轮完成。
第4/5轮完成。
第5/5轮完成。


,model,setting,repeat,order,fit_seconds,predict_seconds,accuracy,macro_f1
0,LogisticRegression,raw,1,1,0.080445,0.000544,0.565833,0.386783
1,SVC,standardized,1,2,0.439884,0.269962,0.793333,0.670190
2,LogisticRegression,standardized,1,3,0.028898,0.000531,0.565833,0.386796
3,DecisionTree,standardized,1,4,0.092570,0.000938,0.495833,0.359136
4,KNN,standardized,1,5,0.001436,0.018614,0.743333,0.602805
5,DecisionTree,raw,1,6,0.093767,0.000625,0.495833,0.359136
6,SVC,raw,1,7,0.409218,0.257152,0.795000,0.669896
7,KNN,raw,1,8,0.000796,0.018128,0.728333,0.598193
8,KNN,standardized,2,1,0.001696,0.014119,0.743333,0.602805
9,SVC,raw,2,2,0.292718,0.180862,0.795000,0.669896


警告总数（含预热）： 0


In [6]:
# 输出：质量、耗时和标准化前后差值；耗时单位秒
metrics = ["accuracy", "macro_f1", "fit_seconds", "predict_seconds"]
grouped = records.groupby(["model", "setting"])[metrics].agg(["mean", "std"])
grouped.columns = [f"{metric}_{stat}" for metric, stat in grouped.columns]
summary = grouped.reset_index()
print("测试集质量：均值与样本标准差（ddof=1）")
display(summary[["model", "setting", "accuracy_mean", "accuracy_std", "macro_f1_mean", "macro_f1_std"]].round(6))
print("训练与整批测试集预测耗时：秒，均值与样本标准差")
display(summary[["model", "setting", "fit_seconds_mean", "fit_seconds_std", "predict_seconds_mean", "predict_seconds_std"]].round(6))
raw = summary.query("setting == 'raw'").set_index("model")
scaled = summary.query("setting == 'standardized'").set_index("model")
changes = pd.DataFrame({
    "accuracy_change_pp": 100*(scaled.accuracy_mean-raw.accuracy_mean),
    "macro_f1_change": scaled.macro_f1_mean-raw.macro_f1_mean,
    "fit_speedup_raw_over_scaled": raw.fit_seconds_mean/scaled.fit_seconds_mean,
    "predict_speedup_raw_over_scaled": raw.predict_seconds_mean/scaled.predict_seconds_mean,
}).reset_index()
print("差值=标准化后−原始；耗时加速比>1表示标准化设置耗时更短。")
display(changes.round(6))
print("重复计时的标准差不是跨数据划分的泛化不确定性；质量标准差为0不代表模型无不确定性。")

测试集质量：均值与样本标准差（ddof=1）


,model,setting,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std
0,DecisionTree,raw,0.495833,0.0,0.359136,0.0
1,DecisionTree,standardized,0.495833,0.0,0.359136,0.0
2,KNN,raw,0.728333,0.0,0.598193,0.0
3,KNN,standardized,0.743333,0.0,0.602805,0.0
4,LogisticRegression,raw,0.565833,0.0,0.386783,0.0
5,LogisticRegression,standardized,0.565833,0.0,0.386796,0.0
6,SVC,raw,0.795000,0.0,0.669896,0.0
7,SVC,standardized,0.793333,0.0,0.670190,0.0


训练与整批测试集预测耗时：秒，均值与样本标准差


,model,setting,fit_seconds_mean,fit_seconds_std,predict_seconds_mean,predict_seconds_std
0,DecisionTree,raw,0.083713,0.007175,0.000520,0.000097
1,DecisionTree,standardized,0.082539,0.007839,0.000806,0.000154
2,KNN,raw,0.000759,0.000163,0.015916,0.003103
3,KNN,standardized,0.001610,0.000276,0.015683,0.003116
4,LogisticRegression,raw,0.070804,0.013408,0.000458,0.000106
5,LogisticRegression,standardized,0.024922,0.004422,0.000521,0.000087
6,SVC,raw,0.335069,0.072751,0.206343,0.036859
7,SVC,standardized,0.406319,0.078164,0.233719,0.038384


差值=标准化后−原始；耗时加速比>1表示标准化设置耗时更短。


,model,accuracy_change_pp,macro_f1_change,fit_speedup_raw_over_scaled,predict_speedup_raw_over_scaled
0,DecisionTree,0.000000,0.000000,1.014221,0.645796
1,KNN,1.500000,0.004613,0.471315,1.014868
2,LogisticRegression,0.000000,0.000012,2.841061,0.879494
3,SVC,-0.166667,0.000294,0.824643,0.882869


重复计时的标准差不是跨数据划分的泛化不确定性；质量标准差为0不代表模型无不确定性。


In [7]:
# 辅助代码：保存结果与机制诊断｜依赖：全部核心单元；无需再次训练
all_predictions = pd.concat(prediction_parts, ignore_index=True)
all_per_class = pd.concat(class_parts, ignore_index=True)
same_predictions = all_predictions.groupby(["model", "setting", "sample_id"]).y_pred.nunique().max() == 1
feature_scale = pd.DataFrame({"feature": np.arange(X_train.shape[1]),
                              "train_mean": X_train.mean(axis=0), "train_std_ddof0": X_train.std(axis=0)})
print("重复运行各设置的预测是否完全一致：", bool(same_predictions))
print("原始训练特征标准差范围：", feature_scale.train_std_ddof0.min(), feature_scale.train_std_ddof0.max())
for filename, table in (("q4_records.csv", records), ("q4_summary.csv", summary),
    ("q4_changes.csv", changes), ("q4_predictions.csv", all_predictions),
    ("q4_per_class.csv", all_per_class), ("q4_feature_scale.csv", feature_scale)):
    table.to_csv(Q4_DIR / filename, index=False, encoding="utf-8-sig")
record = dict(code_version="1.0", source_dir=str(SOURCE_DIR), q3_source=str(Q3_SOURCE),
    data_split_sha256=data_hash, python=sys.version, sklearn=sklearn.__version__, executable=sys.executable,
    platform=platform.platform(), processor=platform.processor(), repeats=REPEATS,
    warmup_per_setting=1, order_seed=ORDER_SEED, thread_limit=THREAD_LIMIT,
    controlled_threadpools=controlled_threadpools, model_parameters=model_parameters,
    preprocessing_parameters=StandardScaler().get_params(), labels=LABELS.tolist(), zero_division=0,
    timing_unit="seconds", std_definition="sample standard deviation ddof=1",
    fit_timing="Pipeline.fit including scaler fit_transform", predict_timing="Pipeline.predict including scaler transform; full 1200-sample batch",
    predictions_identical_across_repeats=bool(same_predictions), output_dir=str(Q4_DIR),
    created_at=datetime.now().astimezone().isoformat())
for filename, content in (("q4_manifest.json", record), ("q4_warnings.json", warning_rows),
                          ("q4_fit_diagnostics.json", diagnostic_rows)):
    (Q4_DIR / filename).write_text(json.dumps(content, ensure_ascii=False, indent=2), encoding="utf-8")
print("保存完成：", Q4_DIR)
print("请发核心代码、质量表、耗时表和差值表截图；有警告请保留全文。")

重复运行各设置的预测是否完全一致： True
原始训练特征标准差范围： 0.9930593528658695 5.562653299921429
保存完成： D:\Python for Data Analysis\notebooks\imbalanced_classification\q4_runs\20261021_122610_cb3b24
请发核心代码、质量表、耗时表和差值表截图；有警告请保留全文。
